# CreditLens — 03 · Modeling

Tune and compare the 6 models on the **15-feature contract** using the **`supervised_models_and_params`
GridSearchCV template**: a list of `{name, model, params}` dicts, looped through `GridSearchCV`, storing
`best_model`, then evaluated with `classification_report` (+ ROC AUC / KS — the right metrics at 8% positives).

Each `model` is a leakage-safe `Pipeline` (impute → [scale] → classifier) from `creditlens.models.registry`,
so every transform is fit on training folds only. **All tried parameters are listed inline below.**

**Speed note:** runs on a 100k stratified subsample so GridSearch finishes in minutes; full-data refit is
`creditlens/pipeline.py` (`make train`) in Phase 4.

## 0 · Setup — data + train/test split

In [1]:
import sys; sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from warnings import filterwarnings
filterwarnings('ignore')

from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, roc_auc_score

from creditlens.data.features import load_or_build_model_matrix, MODEL_FEATURES
from creditlens.models.registry import make_pipeline, make_stacking
from creditlens.config import TARGET, RANDOM_SEED

mat = load_or_build_model_matrix().sample(n=100_000, random_state=RANDOM_SEED)
X, y = mat[MODEL_FEATURES], mat[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_SEED)
print('train', X_train.shape, '| test', X_test.shape, '| positives %.4f' % y.mean())

train (75000, 15) | test (25000, 15) | positives 0.0801


## 1 · Models + parameter grids (the template)
Every parameter we try is listed here. `model` is the preprocessing+classifier pipeline; grid keys target
its `clf` step (e.g. `clf__C`). Imbalance is already handled inside each pipeline
(`class_weight='balanced'` / `scale_pos_weight`).

In [2]:
supervised_models_and_params = [
    {
        'name': 'Logistic Regression',
        'model': make_pipeline('logreg'),
        'params': {
            'clf__C': [0.1, 1, 10],
        },
    },
    {
        'name': 'Random Forest',
        'model': make_pipeline('rf'),
        'params': {
            'clf__n_estimators': [200, 400],
            'clf__max_depth': [15, None],
            'clf__min_samples_leaf': [1, 20],
        },
    },
    {
        'name': 'XGBoost',
        'model': make_pipeline('xgb'),
        'params': {
            'clf__max_depth': [3, 5],
            'clf__learning_rate': [0.03, 0.1],
            'clf__n_estimators': [300, 600],
        },
    },
    {
        'name': 'LightGBM',
        'model': make_pipeline('lgbm'),
        'params': {
            'clf__num_leaves': [31, 63],
            'clf__learning_rate': [0.03, 0.1],
            'clf__n_estimators': [400, 800],
        },
    },
    {
        'name': 'CatBoost',
        'model': make_pipeline('catboost'),
        'params': {
            'clf__depth': [4, 6],
            'clf__learning_rate': [0.03, 0.1],
        },
    },
]

## 2 · Hyperparameter tuning — GridSearchCV loop
Same loop as the template, adapted for binary credit risk: `scoring='roc_auc'` (accuracy is useless at
8% positives) and `StratifiedKFold` (preserves the class ratio per fold).

In [3]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)

for info in supervised_models_and_params:
    grid_search = GridSearchCV(info['model'], info['params'], scoring='roc_auc', cv=cv, n_jobs=-1)
    grid_search.fit(X_train, y_train)
    info['best_model'] = grid_search.best_estimator_
    info['cv_auc'] = grid_search.best_score_
    print(f"{info['name']:22s} best CV AUC={grid_search.best_score_:.4f}  {grid_search.best_params_}")

Logistic Regression    best CV AUC=0.7289  {'clf__C': 0.1}


Random Forest          best CV AUC=0.7355  {'clf__max_depth': 15, 'clf__min_samples_leaf': 20, 'clf__n_estimators': 400}


XGBoost                best CV AUC=0.7391  {'clf__learning_rate': 0.03, 'clf__max_depth': 3, 'clf__n_estimators': 600}


LightGBM               best CV AUC=0.7357  {'clf__learning_rate': 0.03, 'clf__n_estimators': 400, 'clf__num_leaves': 31}


CatBoost               best CV AUC=0.7409  {'clf__depth': 6, 'clf__learning_rate': 0.03}


## 3 · Evaluation — classification_report + AUC/KS on the held-out test set
`classification_report` shows precision/recall/F1 per class at the default 0.5 threshold; for a ranking
model on imbalanced data the headline numbers are **ROC AUC** and **KS** (threshold-free).

In [4]:
def ks_stat(y_true, p):
    order = np.argsort(p); yt = np.asarray(y_true)[order]
    pos, neg = yt.sum(), len(yt) - yt.sum()
    return np.max(np.abs(np.cumsum(yt) / pos - np.cumsum(1 - yt) / neg))

results = []
for info in supervised_models_and_params:
    m = info['best_model']
    y_pred = m.predict(X_test)
    proba = m.predict_proba(X_test)[:, 1]
    auc, ks = roc_auc_score(y_test, proba), ks_stat(y_test, proba)
    results.append({'model': info['name'], 'auc': auc, 'ks': ks, 'gini': 2 * auc - 1})
    print(f"\n=== {info['name']}  (AUC={auc:.4f}  KS={ks:.4f}) ===")
    print(classification_report(y_test, y_pred, target_names=['repaid', 'default']))


=== Logistic Regression  (AUC=0.7294  KS=0.3355) ===
              precision    recall  f1-score   support

      repaid       0.96      0.68      0.80     22998
     default       0.15      0.65      0.24      2002

    accuracy                           0.68     25000
   macro avg       0.55      0.67      0.52     25000
weighted avg       0.89      0.68      0.75     25000




=== Random Forest  (AUC=0.7398  KS=0.3687) ===
              precision    recall  f1-score   support

      repaid       0.95      0.84      0.89     22998
     default       0.20      0.48      0.28      2002

    accuracy                           0.81     25000
   macro avg       0.58      0.66      0.59     25000
weighted avg       0.89      0.81      0.84     25000


=== XGBoost  (AUC=0.7431  KS=0.3721) ===
              precision    recall  f1-score   support

      repaid       0.96      0.69      0.80     22998
     default       0.16      0.68      0.26      2002

    accuracy                           0.69     25000
   macro avg       0.56      0.69      0.53     25000
weighted avg       0.90      0.69      0.76     25000




=== LightGBM  (AUC=0.7405  KS=0.3639) ===
              precision    recall  f1-score   support

      repaid       0.96      0.73      0.83     22998
     default       0.17      0.63      0.27      2002

    accuracy                           0.72     25000
   macro avg       0.56      0.68      0.55     25000
weighted avg       0.89      0.72      0.78     25000


=== CatBoost  (AUC=0.7428  KS=0.3652) ===
              precision    recall  f1-score   support

      repaid       0.96      0.69      0.81     22998
     default       0.16      0.67      0.26      2002

    accuracy                           0.69     25000
   macro avg       0.56      0.68      0.53     25000
weighted avg       0.90      0.69      0.76     25000



## 4 · Stacking ensemble
Blends the 3 boosters via a LogReg meta-learner (out-of-fold base predictions). Fit on train, eval on test.

In [5]:
stack = make_stacking().fit(X_train, y_train)
proba = stack.predict_proba(X_test)[:, 1]
auc, ks = roc_auc_score(y_test, proba), ks_stat(y_test, proba)
results.append({'model': 'Stacking', 'auc': auc, 'ks': ks, 'gini': 2 * auc - 1})
print(f'Stacking  AUC={auc:.4f}  KS={ks:.4f}')
print(classification_report(y_test, stack.predict(X_test), target_names=['repaid', 'default']))

Stacking  AUC=0.7427  KS=0.3724


              precision    recall  f1-score   support

      repaid       0.92      1.00      0.96     22998
     default       0.00      0.00      0.00      2002

    accuracy                           0.92     25000
   macro avg       0.46      0.50      0.48     25000
weighted avg       0.85      0.92      0.88     25000



## 5 · Leaderboard

In [6]:
board = pd.DataFrame(results).sort_values('auc', ascending=False).reset_index(drop=True)
board.round(4)

,model,auc,ks,gini
0,XGBoost,0.7431,0.3721,0.4862
1,CatBoost,0.7428,0.3652,0.4855
2,Stacking,0.7427,0.3724,0.4854
3,LightGBM,0.7405,0.3639,0.4811
4,Random Forest,0.7398,0.3687,0.4796
5,Logistic Regression,0.7294,0.3355,0.4589


# Notebook summary & key insights

## Task
Tune and compare 6 models (LogReg, RandomForest, XGBoost, LightGBM, CatBoost, Stacking) on the 15-feature
contract via the `supervised_models_and_params` GridSearch template; pick the candidate to calibrate & serve.

## Setup
- Features: `load_or_build_model_matrix` (cached). 100k stratified subsample, 75/25 train/test.
- Each model = leakage-safe `Pipeline` (impute → [scale] → clf); grids listed inline in section 1.
- Tuning: `GridSearchCV(scoring='roc_auc', cv=StratifiedKFold(3))`; eval: `classification_report` + AUC/KS.

## Findings
- _From the leaderboard:_ gradient-boosted trees + stacking lead (~0.74 AUC); LogReg is the interpretable
  floor (~0.73). Best CV params printed in section 2.
- `classification_report` recall on the `default` class is modest at the 0.5 threshold — expected; the
  operating threshold is chosen later from the risk bands, not fixed at 0.5.

## Insights & Recommendations
- **Insight:** AUC ~0.74 is the ceiling for this 15-feature *form contract* — ~0.02–0.03 below a full
  146-feature model, the price of a model the frontend form can actually feed.
- **Insight:** Tune on ROC AUC, not accuracy; report `classification_report` for intuition but don't select on it.
- **Recommendation:** Don't read decisions off the 0.5-threshold report — set the threshold from the cost of
  false approve vs false reject (Phase 4 bands).
- **Recommendation:** Refit the chosen config on full data in `pipeline.py`; calibrate before serving.

## Next
Notebook **04 · Evaluation** — calibrate the best model (isotonic), ROC / reliability / lift, finalize the
model-card numbers, save the served artifact + metadata.